# BoldSearch Kaggle dev machine

Kaggle chỉ là máy GPU; mọi thao tác tiếp theo bạn làm từ VS Code attach vào (URL + password in ở cell cuối).

Attach vào Input: (1) dataset keyframes, (2) dataset private `.env` do CI publish — đã chứa path Kaggle (`KEYFRAMES_DIR` = `/kaggle/input/<slug>/keyframes`). Settings: GPU + Internet bật, rồi **Run All**.

Sau khi mở VS Code: terminal chạy BE — `cd app/backend && uv run uvicorn main:app --host 127.0.0.1 --port 8000` (repo tự validate: `AppConfig` parse env, lifespan kết nối Zilliz).

In [ ]:
import json
import os
import shutil
import subprocess
from pathlib import Path

runtime = Path("/kaggle/working/boldsearch-runtime")
runtime.mkdir(exist_ok=True)

print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip() or "no GPU — BEiT3 sẽ fail ở startup")

env_candidates = [e / ".env" for e in Path("/kaggle/input").iterdir() if (e / ".env").is_file()]
assert len(env_candidates) == 1, f"attach đúng 1 env dataset của CI, thấy: {env_candidates}"
env_file = env_candidates[0]


def env_value(key: str) -> str:
    for line in env_file.read_text(encoding="utf-8").splitlines():
        if line.startswith(key + "="):
            value = line.split("=", 1)[1].strip()
            return json.loads(value) if value.startswith('"') else value
    return ""


REPO_URL = env_value("REPO_URL")
assert REPO_URL, "env dataset thiếu REPO_URL"
repo = Path("/kaggle/working/BoldSearch")

askpass = runtime / "git-askpass.sh"
askpass.write_text(
    '#!/bin/sh\ncase "$1" in *Username*) echo x-access-token ;; *) echo "$BOLDSEARCH_GIT_TOKEN" ;; esac\n',
    encoding="utf-8",
)
askpass.chmod(0o700)
git_env = {**os.environ, "GIT_ASKPASS": str(askpass), "GIT_TERMINAL_PROMPT": "0", "BOLDSEARCH_GIT_TOKEN": env_value("GH_PAT")}

if (repo / ".git").is_dir():
    subprocess.run(["git", "-C", str(repo), "fetch", "--depth", "1", "origin"], env=git_env, check=True)
    subprocess.run(["git", "-C", str(repo), "reset", "--hard", "FETCH_HEAD"], env=git_env, check=True)
else:
    if repo.exists():
        shutil.rmtree(repo)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo)], env=git_env, check=True)
askpass.unlink()

# env dataset = backend .env nguyên văn, đã chứa path Kaggle từ secret
shutil.copy(env_file, repo / "app" / "backend" / ".env")
print("commit:", subprocess.check_output(["git", "-C", str(repo), "rev-parse", "--short", "HEAD"], text=True).strip())

In [ ]:
!pip install -q uv
!uv sync --frozen --directory /kaggle/working/BoldSearch/app/backend
print("env ready: /kaggle/working/BoldSearch/app/backend/.venv")

In [ ]:
import os
import re
import secrets
import subprocess
import time
from urllib.request import urlretrieve

code_server = runtime / "code-server"
if not code_server.is_file():
    urlretrieve(
        "https://github.com/coder/code-server/releases/latest/download/code-server-linux-amd64.tar.gz",
        runtime / "code-server.tar.gz",
    )
    subprocess.run(["tar", "-xzf", str(runtime / "code-server.tar.gz"), "-C", str(runtime)], check=True)
    next(runtime.glob("code-server-*/bin/code-server")).rename(code_server)

password = secrets.token_hex(8)
with (runtime / "code-server.log").open("ab") as handle:
    subprocess.Popen(
        [str(code_server), "--bind-addr", "127.0.0.1:8080", "--auth", "password",
         "--disable-telemetry", "--disable-workspace-trust", str(repo)],
        env={**os.environ, "PASSWORD": password},
        stdout=handle, stderr=subprocess.STDOUT, start_new_session=True,
    )

cloudflared = runtime / "cloudflared"
if not cloudflared.is_file():
    urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        cloudflared,
    )
    cloudflared.chmod(0o755)
with (runtime / "tunnel.log").open("ab") as handle:
    subprocess.Popen(
        [str(cloudflared), "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8080"],
        stdout=handle, stderr=subprocess.STDOUT, start_new_session=True,
    )

for _ in range(60):
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", (runtime / "tunnel.log").read_text(errors="replace"))
    if match:
        print(f"TUNNEL_URL={match.group(0)}")
        print(f"VS Code password: {password}")
        print("Mở URL -> nhập password -> terminal: cd app/backend && uv run uvicorn main:app --host 127.0.0.1 --port 8000")
        break
    time.sleep(2)
else:
    raise RuntimeError("tunnel không public URL kịp timeout")
while True:
    time.sleep(60)